Notebook to create emulations of Annual Maximum Daily Precipitation **Rx1day**

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

In [2]:
import mesmer

In [5]:
# add pathway to folders 1 level higher (i.e., to mesmer and configs)
import os
import sys
import time as tm

import joblib
import matplotlib.pyplot as plt

sys.path.append("/net/argon/landclim2/lpierini/mesmer/")

import mesmer.mesmer_x.train_l_distrib_mesmerx as mesmer_x_train
import mesmer.mesmer_x.train_utils_mesmerx as mesmer_x_train_utils

from mesmer.mesmer_x import load_cmip_mesmerx

from mesmer.create_emulations import create_emus_g, create_emus_gt, create_emus_gv, create_emus_l, create_emus_lt, create_emus_lv
from mesmer.calibrate_mesmer import train_gt, train_gv, train_lt, train_lv

# load in MESMER-X configurations used in this script
from mesmer.mesmer_x.temporary_config_all import ConfigMesmerX
from mesmer.mesmer_x.temporary_support import load_inputs_MESMERx
from mesmer.utils import separate_hist_future

In [16]:
import importlib

importlib.reload(mesmer_x_train_utils)
importlib.reload(mesmer_x_train)
importlib.reload(mesmer)

<module 'mesmer' from '/net/argon/landclim2/lpierini/mesmer/mesmer/__init__.py'>

## Settings

In [6]:
 # ==============================================================
# 0. OPTIONS FOR THE SCRIPT
# ==============================================================
# variables to represent
targ = "rx1day"  # rx1day, txx, mrso, fwils, fwisa, fwixd, fwixx, mrso_minmon mrsomean??!
pred = "tas"
sub_pred = None  # 'hfds' hfds | (pr)

# options for server
# To control if run everything or using a bunch of processes
run_on_exo = False
# ==============================================================
# ==============================================================

### Input Output

In [7]:
# ==============================================================
# 1. PREPARATION OF MESMER-X
# ==============================================================
# short preparation
# Priority of this script on the server. Value in [-20,19], default at 0, higher is nicer to others
if run_on_exo:
    #os.nice(19)
    os.nice(5)
subindex_csl = int(sys.argv[1]) if run_on_exo else None
runs_per_process = 3

# paths
path_save_figures = "/home/lpierini/Results/mesmer/figs/" + targ
path_save_results = "/home/lpierini/Results/mesmer/res/" + targ

# configuration
gen = 6
dir_cmipng = "/net/atmos/data/cmip" + str(gen) + "-ng/"
# /net/ch4/data/cmip6-Next_Generation/mrso/ann/g025

In [8]:
dir_cmip_X = {
    'rx1day': '/net/argon/landclim2/lpierini/annual_indicators/rx1day', 
    "txx": "/net/cfc/landclim1/mathause/projects/IPCC_AR6_CH11/IPCC_AR6_CH11/data/cmip6/tasmax/txx_regrid",
    "mrso": "/net/ch4/data/cmip6-Next_Generation/mrso/ann/g025",
    "mrsomean": "/landclim/yquilcaille/annual_indicators/mrsomean",
    "mrso_minmon": "/landclim/yquilcaille/annual_indicators/mrso_minmon/ann/g025",
    "fwixx": "/landclim_nobackup/yquilcaille/FWI_CMIP6/hurs_tasmax_sfcWind_pr/Drying-NSHeq_Day-continuous_Owinter-wDC/regridded/fwixx/ann/g025",
    "fwisa": "/landclim_nobackup/yquilcaille/FWI_CMIP6/hurs_tasmax_sfcWind_pr/Drying-NSHeq_Day-continuous_Owinter-wDC/regridded/fwisa/ann/g025",
    "fwixd": "/landclim_nobackup/yquilcaille/FWI_CMIP6/hurs_tasmax_sfcWind_pr/Drying-NSHeq_Day-continuous_Owinter-wDC/regridded/fwixd/ann/g025",
    "fwils": "/landclim_nobackup/yquilcaille/FWI_CMIP6/hurs_tasmax_sfcWind_pr/Drying-NSHeq_Day-continuous_Owinter-wDC/regridded/fwils/ann/g025",
}[targ]
# '/net/cfc/landclim1/mathause/projects/IPCC_AR6_CH11/IPCC_AR6_CH11/data/cmip6/mrso/sm_annmean'

# observations
dir_obs = "/net/exo/landclim/yquilcaille/mesmer-x/data/observations/"
# auxiliary data
dir_aux = "/net/exo/landclim/yquilcaille/mesmer-x/data/auxiliary/"
dir_mesmer_params = "/net/exo/landclim/yquilcaille/mesmer-x/calibrated_parameters/"
dir_mesmer_emus = "/net/exo/landclim/yquilcaille/mesmer-x/emulations/"
# emulation statistics
dir_stats = "/net/exo/landclim/yquilcaille/mesmer-x/statistics/"
# plots
dir_plots = "/net/exo/landclim/yquilcaille/mesmer-x/plots/"

In [9]:
cfg = ConfigMesmerX(
    gen=gen,
    paths={
        "dir_cmipng": dir_cmipng,
        "dir_cmip_X": dir_cmip_X,
        "dir_obs": dir_obs,
        "dir_aux": dir_aux,
        "dir_mesmer_params": dir_mesmer_params,
        "dir_mesmer_emus": dir_mesmer_emus,
        "dir_stats": dir_stats,
        "dir_plots": dir_plots,
    },
    esms="all",
)

# make paths if not existing
for path in [path_save_results, path_save_figures]:
    if not os.path.exists(path):
        os.makedirs(path)

In [10]:
#let us start with these
cfg.scenarios = ['h-ssp585',
 'h-ssp370',
 'h-ssp245',
 'h-ssp126',
 'h-ssp119']

### Preparation

In [11]:
esms_test = ['ACCESS-CM2',
             #'ACCESS-ESM1-5',
             #'CanESM5',
             #'CESM2',
             #'CESM2-WACCM',
             #'CNRM-ESM2-1',
             #'FGOALS-g3',
             #'HadGEM3-GC31-LL',
             #'HadGEM3-GC31-MM',
             #'MPI-ESM1-2-HR',
             #'MPI-ESM1-2-LR',
             #'MRI-ESM2-0',
             #'NESM3',
             #'NorESM2-LM',
             #'UKESM1-0-LL',
             #'NorESM2-MM',
             #'CMCC-CM2-SR5',
             #'CNRM-CM6-1',
             #'E3SM-1-1',
             #'FGOALS-f3-L',
             #'IPSL-CM6A-LR',
            #'AWI-CM-1-1-MR',
            #'MCM-UA-1-0',
            #'FIO-ESM-2-0',
            ]

In [13]:
stats_models = ["genextreme(loc=c1 + c2 * __GMT_t__, scale=c3 + c4 * __GMT_t__, c=c7)",
                #"genextreme(loc=c1 + c2 * __GMT_t__+ c3 * __GMTv_t__, scale=c4 + c5 * __GMT_t__, c=c7)",
                ]

#laebl of the statistical model chosen
stats_labels = {#stats_models[0]: 'const',
                #stats_models[0]: 'lin_loc', 
                stats_models[0]: 'lin_loc_scale', 
                #stats_models[1]: 'quad_loc',
                }

In [15]:
expr_mix = 'genextreme(loc=c1 + c2 * __GMT_t__, scale=c3 , c=c7)'
#expr_mix = 'genextreme(loc=c1 + c2 * __GMT_t__+ c3 * __GMTv_t__, scale=c4 , c=c7)'
#expr_mix = 'genextreme(loc=c1 + c2 * __GMTv_t__, scale=c4 , c=c7)'

#label for mixture
expr_lab_mix = 'mix_lin_loc'

In [17]:
#optimization settings

opt_save = False #Not Saving Results

nbins_weights = 20
phys_thres_on = True #Rectification is active
weighting_choice = True
mix_on = True #Mixture option is active
p_time_on = True #!!!!!!
max_p_mix_val = None
MAT_min = 0.12 #threshold on which to flag gridpoints for mixture fit
cmin = -0.52 #minimum shape value, note that vor shape above 0.5, the variance (and mean) are not defined
smod = 'glob'
#smod = 'glob_mixvar'

### Training

In [18]:
#import pickle
# --------------------------------------------------------------
# --------------------------------------------------------------
#ESM selection
# --------------------------------------------------------------

import sys

## Redirect output to a file
log_file = open("output_log.txt", "w")
sys.stdout = log_file

start_time = tm.time()

for esm in esms_test:
    try:
        esms = [esm]  # cfg.esms
        #print(esms)
        
        # --------------------------------------------------------------
        # --------------------------------------------------------------
        
        if run_on_exo:
            esms = esms[
                subindex_csl * runs_per_process : (subindex_csl + 1) * runs_per_process
            ]
            
        # --------------------------------------------------------------
        # preparing data
        # --------------------------------------------------------------
        (
            time,
            PRED,
            SUB_PRED,
            reg_dict,
            ls,
            wgt_g,
            lon,
            lat,
            land_targ,
            land_pred,
            phi_gc,
            #ind,
            #gp2reg,
            #ww_reg,
            used_esms,
            dico_gps_nan,
        ) = load_inputs_MESMERx(cfg, [targ, pred, sub_pred], esms)
        
        
        # --------------------------------------------------------------
        #DATA FORMATTING
        # --------------------------------------------------------------
        
        # esm test
        esm = esms[0]
        
        # variable
        land_targ_s, time_s = separate_hist_future(land_targ[esm], time[esm], cfg)
        
        # predictor GMT at t
        pred_s, time_s = separate_hist_future(PRED[esm], time[esm], cfg)

        #Global trend only of GMT
        preds_gt = {"time": time[esm]}
        params_gt_pred = train_gt(PRED[esm], 'tas', esm, time[esm], cfg, save_params=False)
        emus_gt_pred = create_emus_gt(
                    params_gt_pred, preds_gt, cfg, concat_h_f=True, save_emus=False
                )
        gt_pred_s = create_emus_gt(
            params_gt_pred, preds_gt, cfg, concat_h_f=False, save_emus=False
        )

        
        pred_s_m = {ss: np.tile(gt_pred_s[ss], (land_targ_s[ss].shape[0], 1)) for ss in gt_pred_s.keys()} #smoothed GMT, same for every member
        
        # predictor GMT at t-1
        tmp_pred_s = {
            scen: np.hstack([pred_s_m[scen][:, 0, np.newaxis], pred_s_m[scen][:, :-1]])
            for scen in pred_s_m.keys()
        }
        #tmp_pred_s, _ = separate_hist_future(tmp_pred, time[esm], cfg)

        #global variability

        gv_novolc_pred = {scen: PRED[esm][scen] - emus_gt_pred[scen] for scen in emus_gt_pred.keys()}
        gv_novolc_pred_s, time_s = separate_hist_future(gv_novolc_pred, time[esm], cfg)

    
        # preparing predictors (GMT)
        predictors = []
        for scen in pred_s.keys():
            predictors.append((xr.Dataset(), scen))
            predictors[-1][0]["GMT_t"] = xr.DataArray(
                pred_s_m[scen],
                coords={"member": np.arange(pred_s[scen].shape[0]), "time": time_s[scen]},
                dims=(
                    "member",
                    "time",
                ),
            )
            #variability, if used
            predictors[-1][0]["GMTv_t"] = xr.DataArray(
                gv_novolc_pred_s[scen],
                coords={"member": np.arange(pred_s[scen].shape[0]), "time": time_s[scen]},
                dims=(
                    "member",
                    "time",
                ),
            )
            #predictors[-1][0]["GMT_tm1"] = xr.DataArray(
            #    tmp_pred_s[scen],
            #    coords={"member": np.arange(pred_s[scen].shape[0]), "time": time_s[scen]},
           #     dims=(
            #        "member",
            #        "time",
            #    ),
            #)
        
        
        # preparing target
        target = []
        for scen in land_targ_s.keys():
            target.append((xr.Dataset(), scen))
            target[-1][0][targ] = xr.DataArray(
                land_targ_s[scen],
                coords={
                    "member": np.arange(land_targ_s[scen].shape[0]),
                    "time": time_s[scen],
                    "gridpoint": np.arange(land_targ_s[scen].shape[2]),
                },
                dims=(
                    "member",
                    "time",
                    "gridpoint",
                ),
            )
        
        
        # --------------------------------------------------------------
        #TRAINING
        # --------------------------------------------------------------
        
        #Expression selection
        expr_name = "gev"
    
        
        for expr in stats_models:
            #for expr_lab in stats_labels:
            expr_lab = stats_labels[expr]
            #expr = "genextreme(loc=c1 + c2 * __GMT_t__, scale=c3 + c4 * __GMT_t__, c=c7)"
            #expr_lab = 'lin_loc'
            
            
            print(expr_lab)
            
            if expr_lab == 'lin_loc':
                for i in range(len(predictors)):
                    scen = predictors[i][1]
                    #predictors[i][0].GMT_t.to_netcdf(path_save_results+ '/GMT_' + esm + '_' + scen + '.nc')



            #training conditional distributions following 'expr' in all grid points
            xr_coeffs_distrib, xr_qual = mesmer_x_train.xr_train_distrib(
                        predictors=predictors,
                        target=target,
                        target_name=targ,
                        expr=expr,
                        expr_mix = expr_mix,
                        expr_name=expr_name,
                        p_time = p_time_on, #!!!!!!!!!!!
                        max_p_mix = max_p_mix_val,
                        option_2ndfit=False,
                        option_2ndfit_mix = mix_on, #!!!!!!!!!!!
                        r_gasparicohn_2ndfit=500,
                        scores_fit=["func_optim", "rectification", "rect_loc", "rect_std", "mean_above_threshold_95", "mean_above_threshold_995", "NLL", "BIC", "spread_clusters", "small_cluster", "scaling_mean", "silhouette", "count_995","count_95", "count_50", "count_05"],
                        boundaries_params={'c': [cmin, 0.4], 'loc': [-np.inf, np.inf], 'scale': [0, np.inf]},
                        boundaries_params_mix={'c': [-0.25,0.7], 'loc': [-np.inf, np.inf], 'scale': [0, np.inf]},
                        options_optim = {"weighted_NLL": weighting_choice, "phys_thres_on": phys_thres_on, "nbins_weights":nbins_weights},
                        n_jobs=30,
                        MAT_min = MAT_min
                    )

            xr_coeffs_distrib.attrs["weighting"] = str(weighting_choice)
            xr_qual.attrs["weighting"] = str(weighting_choice)
            xr_coeffs_distrib.attrs["nbins_weight"] = nbins_weights
            xr_qual.attrs["nbins_weight"] = nbins_weights
            xr_coeffs_distrib.attrs["phys_thres_on"] = str(phys_thres_on)
            xr_qual.attrs["phys_thres_on"] = str(phys_thres_on)
            xr_coeffs_distrib.attrs["mixing"] = str(mix_on)
            xr_qual.attrs["mixing"] = str(mix_on)
            xr_coeffs_distrib.attrs["p_time_dep"] = str(p_time_on)
            xr_qual.attrs["p_time_dep"] = str(p_time_on)
            xr_coeffs_distrib.attrs["MAT_min"] = MAT_min
            xr_qual.attrs["MAT_min"] = MAT_min
            xr_coeffs_distrib.attrs["max_p0_mix"] = str(max_p_mix_val)
            xr_qual.attrs["max_p0_mix"] = str(max_p_mix_val)

            if opt_save == True:
                if (phys_thres_on == True) and (mix_on == True) and (p_time_on == True):
                    xr_coeffs_distrib.to_netcdf(path_save_results+ '/coeffs_training_'+smod+'_weights_rect_mix_'+targ+'_' + esm + '_'+expr_lab+'_' + expr_lab_mix+'MAT' + str(MAT_min).replace('.', '') + '.nc')
                    xr_qual.to_netcdf(path_save_results+ '/qual_training_'+smod+'_weights_rect_mix_'+targ+'_' + esm +'_'+expr_lab+'_' + expr_lab_mix+'MAT' + str(MAT_min).replace('.', '') +'.nc')
                elif (phys_thres_on == True) and (mix_on == True) and (p_time_on == False):
                    xr_coeffs_distrib.to_netcdf(path_save_results+ '/coeffs_training_'+smod+'_weights_rect_mixconst_'+targ+'_' + esm + '_'+expr_lab+'_' + expr_lab_mix+'MAT' + str(MAT_min).replace('.', '') + '.nc')
                    xr_qual.to_netcdf(path_save_results+ '/qual_training_'+smod+'_weights_rect_mixconst_'+targ+'_' + esm +'_'+expr_lab+'_' + expr_lab_mix+'MAT' + str(MAT_min).replace('.', '') +'.nc')
                elif (phys_thres_on == False) and (mix_on == False):
                    xr_coeffs_distrib.to_netcdf(path_save_results+ '/coeffs_training_'+smod+'_weights_norect_nomix_'+targ+'_' + esm + '_'+expr_lab +'.nc')
                    xr_qual.to_netcdf(path_save_results+ '/qual_training_'+smod+'_weights_norect_nomix_'+targ+'_' + esm +'_'+expr_lab+'.nc')
                elif (phys_thres_on == True) and (mix_on == False):
                    xr_coeffs_distrib.to_netcdf(path_save_results+ '/coeffs_training_'+smod+'_weights_rect_nomix_'+targ+'_' + esm + '_'+expr_lab+'.nc')
                    xr_qual.to_netcdf(path_save_results+ '/qual_training_'+smod+'_weights_rect_nomix_'+targ+'_' + esm +'_'+expr_lab+'.nc')
                else:
                    print('Files not saved, check rectification and mixture options')
                
            
            #
            
            #with open(path_save_results+'/dict_resid_'+ esm + '_' +expr_lab + '.pkl', 'wb') as f:
            #    pickle.dump(dict_resid, f)
    except Exception as e:
        # If an error occurs, log the esm and the error
        print(f"Error processing {esm}: {e}")
        problem = e
        problematic_esms.append(esm)


# End time
end_time = tm.time()

# Execution time in seconds
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time:.6f} seconds")

/landclim2/lpierini/mesmer/mesmer/io/load_constant_files.py:178: FutureWarning: ``reg_type`` no longer has any effect.
  warnings.warn("``reg_type`` no longer has any effect.", FutureWarning)
/landclim2/lpierini/mesmer/mesmer/utils/select.py:67: FutureWarning: Passing `reg_dict` no longer has an effect.
  warnings.warn("Passing `reg_dict` no longer has an effect.", FutureWarning)
/landclim2/lpierini/mesmer/mesmer/create_emulations/create_emus_gt.py:17: FutureWarning: 'create_emus_gt' has been renamed to `gather_gt_data`
  warnings.warn(


In [20]:
xr_coeffs_distrib;

## Emulation Generation

### load data

In [21]:
esms_test = ['ACCESS-CM2',
 #'ACCESS-ESM1-5',
 #'AWI-CM-1-1-MR',
 #'CanESM5',
 #'CESM2',
 #'CESM2-WACCM',
 #'CMCC-CM2-SR5',
 #'CNRM-CM6-1',
 #'CNRM-CM6-1-HR',
 #'CNRM-ESM2-1',
 #'E3SM-1-1',
 #'FGOALS-f3-L',
 #'FGOALS-g3',
 #'FIO-ESM-2-0',
 #'HadGEM3-GC31-LL',
 #'HadGEM3-GC31-MM',
 #'IPSL-CM6A-LR',
 #'MCM-UA-1-0',
 #'MPI-ESM1-2-HR',
 #'MPI-ESM1-2-LR',
 #'MRI-ESM2-0',
 #'NESM3',
 #'NorESM2-LM',
 #'NorESM2-MM',
 #'UKESM1-0-LL'
            ]

In [22]:
expr = 'genextreme(loc=c1 + c2 * __GMT_t__, scale=c3 + c4 * __GMT_t__ , c=c7)'
expr_lab = 'lin_loc_scale'

In [23]:
expr_mix = 'genextreme(loc=c1 + c2 * __GMT_t__, scale=c3 , c=c7)'
expr_lab_mix = 'mix_lin_loc'

In [24]:
# Initialize the variables before the loop
xr_coeffs_distrib0 = None
xr_qual0 = None

for i, esm in enumerate(esms_test):
    xr_coeffs_distrib = xr.open_dataset(path_save_results+ '/coeffs_training_glob_weights_rect_mix_'+targ+'_' + esm + '_'+expr_lab+'_' + expr_lab_mix+'MAT012.nc')
    xr_coeffs_distrib = xr_coeffs_distrib.expand_dims(model = [esm])

    xr_qual = xr.open_dataset(path_save_results+ '/qual_training_glob_weights_rect_mix_'+targ+'_' + esm + '_'+expr_lab+'_' + expr_lab_mix+'MAT012.nc')
    xr_qual = xr_qual.expand_dims(model = [esm])
    
    if i>0:
        xr_coeffs_distrib1 = xr.concat((xr_coeffs_distrib, xr_coeffs_distrib0), dim = 'model')
        xr_coeffs_distrib0 = xr_coeffs_distrib1

        xr_qual1 = xr.concat((xr_qual, xr_qual0), dim = 'model')
        xr_qual0 = xr_qual1
    else:
        xr_coeffs_distrib0 = xr_coeffs_distrib
        xr_qual0 = xr_qual

if len(esms_test) == 1:
    xr_coeffs_distrib_rect_mix_glob = xr_coeffs_distrib0
    xr_qual_rect_mix_glob = xr_qual0
else:
    xr_coeffs_distrib_rect_mix_glob = xr_coeffs_distrib1
    xr_qual_rect_mix_glob = xr_qual1

## Normalized Residuals and Emulations

In [85]:
dict_res_mm = {}
#dict_res_0_mm = {}

In [86]:
#CAREFUL: I need the predictors from the corresponding models!!!

esmodel = 'ACCESS-CM2'

# --------------------------------------------------------------
# preparing data
# --------------------------------------------------------------
(
    time,
    PRED,
    SUB_PRED,
    reg_dict,
    ls,
    wgt_g,
    lon,
    lat,
    land_targ,
    land_pred,
    phi_gc,
    #ind,
    #gp2reg,
    #ww_reg,
    used_esms,
    dico_gps_nan,
) = load_inputs_MESMERx(cfg, [targ, pred, sub_pred], [esmodel])


# --------------------------------------------------------------
#DATA FORMATTING
# --------------------------------------------------------------

# esm test
esm = esmodel

# variable
land_targ_s, time_s = separate_hist_future(land_targ[esm], time[esm], cfg)

# predictor GMT at t
pred_s, time_s = separate_hist_future(PRED[esm], time[esm], cfg)

#Global trend only of GMT
preds_gt = {"time": time[esm]}
params_gt_pred = train_gt(PRED[esm], 'tas', esm, time[esm], cfg, save_params=False)

gt_pred_s = create_emus_gt(
    params_gt_pred, preds_gt, cfg, concat_h_f=False, save_emus=False
)


pred_s_m = {ss: np.tile(gt_pred_s[ss], (land_targ_s[ss].shape[0], 1)) for ss in gt_pred_s.keys()} #smoothed GMT, same for every member

# predictor GMT at t-1
tmp_pred_s = {
    scen: np.hstack([pred_s_m[scen][:, 0, np.newaxis], pred_s_m[scen][:, :-1]])
    for scen in pred_s_m.keys()
}
#tmp_pred_s, _ = separate_hist_future(tmp_pred, time[esm], cfg)

#global trend emulations
emus_gt_pred = create_emus_gt(
            params_gt_pred, preds_gt, cfg, concat_h_f=True, save_emus=False
        )

#global variability

gv_novolc_pred = {scen: PRED[esm][scen] - emus_gt_pred[scen] for scen in emus_gt_pred.keys()}
gv_novolc_pred_s, time_s = separate_hist_future(gv_novolc_pred, time[esm], cfg)


# preparing predictors (GMT)
predictors = []
for scen in pred_s.keys():
    predictors.append((xr.Dataset(), scen))
    predictors[-1][0]["GMT_t"] = xr.DataArray(
        pred_s_m[scen]-np.nanmin(pred_s_m['hist']),   #I want to start from 0 anomaly (that's how i took the training)
        coords={"member": np.arange(pred_s[scen].shape[0]), "time": time_s[scen]},
        dims=(
            "member",
            "time",
        ),
    )
    predictors[-1][0]["GMTv_t"] = xr.DataArray(
        gv_novolc_pred_s[scen],
        coords={"member": np.arange(pred_s[scen].shape[0]), "time": time_s[scen]},
        dims=(
            "member",
            "time",
        ),
    )
    #predictors[-1][0]["GMT_tm1"] = xr.DataArray(
    #    tmp_pred_s[scen],
    #    coords={"member": np.arange(pred_s[scen].shape[0]), "time": time_s[scen]},
   #     dims=(
    #        "member",
    #        "time",
    #    ),
    #)

# preparing target
target = []
for scen in land_targ_s.keys():
    target.append((xr.Dataset(), scen))
    target[-1][0][targ] = xr.DataArray(
        land_targ_s[scen],
        coords={
            "member": np.arange(land_targ_s[scen].shape[0]),
            "time": time_s[scen],
            "gridpoint": np.arange(land_targ_s[scen].shape[2]),
        },
        dims=(
            "member",
            "time",
            "gridpoint",
        ),
    )

#MAT01
xr_coeffs_distrib = xr_coeffs_distrib_rect_mix_glob.sel(model = esmodel)

xr_qual = xr_qual_rect_mix_glob.sel(model = esmodel)

expression_fit = mesmer_x_train.Expression(expr, expr_lab)
#distrib = expression_fit.evaluate(xr_coeffs_distrib, predictors[1][0], forced_shape=target[1][0]['rx1day'].dims)

xr_coeffs_distrib_m = xr_coeffs_distrib.drop_vars(expression_fit.coefficients_list)
xr_coeffs_distrib_mix = xr_coeffs_distrib_m.rename({var: var[:-1] for var in xr_coeffs_distrib_m.data_vars if var.endswith('m')})

#xr_coeffs_distrib_bs = xr_coeffs_distrib_no_rm.sel(model = esmodel)
#xr_qual_bs = xr_qual_no_rm.sel(model = esmodel)

transf_target = mesmer_x_train_utils.probability_integral_transform(  # noqa: F841
    data=target, 
    target_name = targ,
    expr_start=expr,
    expr_start_mix=expr_mix,
    coeffs_start=xr_coeffs_distrib,
    coeffs_start_mix=xr_coeffs_distrib_mix,
    qual_start = xr_qual,
    preds_start=predictors,
    expr_end="norm(loc=0, scale=1)",
)


#transf_target_0 = mesmer_x_train_utils.probability_integral_transform(  # noqa: F841
#    data=target, 
#    target_name = targ,
#    expr_start=expr,
#    coeffs_start=xr_coeffs_distrib_bs,
#    preds_start=predictors,
#    expr_end="norm(loc=0, scale=1)",
#)

#Transforming to dictionary containing scenarios

transf_target_dict = {targ: 
                      {str(transf_target[i][1]): transf_target[i][0] for i in range(len(transf_target))}
                     }
#transf_target_dict_0 = {targ: 
#                      {str(transf_target_0[i][1]): transf_target_0[i][0] for i in range(len(transf_target_0))}
#                     }

dict_resid = {}
#dict_resid_0 = {}

#maybe save only the residuals for historical and ssp585 as netcdf to save space
# Iterate through the outer dictionary (rx1day)
for scenario, np_array in transf_target_dict['rx1day'].items():
    # Assuming the shape is (member, time, gridpoint)
    members, timesteps, gridpoints = np_array.shape
    
    # Create DataArray with dimensions and coordinates
    dict_resid[scenario] = xr.DataArray(
        np_array,
        dims=['member', 'time', 'gridpoint'],
        coords={
            'member': np.arange(members),
            'time': np.arange(timesteps),
            'gridpoint': np.arange(gridpoints)
        },
        name=scenario
    )

#NO RECT NO MIX case
#for scenario, np_array in transf_target_dict_0['rx1day'].items():
    # Assuming the shape is (member, time, gridpoint)
#    members, time, gridpoints = np_array.shape
    
    # Create DataArray with dimensions and coordinates
#    dict_resid_0[scenario] = xr.DataArray(
#        np_array,
#        dims=['member', 'time', 'gridpoint'],
#        coords={
#            'member': np.arange(members),
#            'time': np.arange(time),
#            'gridpoint': np.arange(gridpoints)
#        },
#        name=scenario
#    )

#dict_res_mm[esmodel] = dict_resid
#dict_res_0_mm[esmodel] = dict_resid_0

dict_res_mm[esmodel] = dict_resid

/net/argon/landclim2/lpierini/mesmer/mesmer/io/load_constant_files.py:178: FutureWarning: ``reg_type`` no longer has any effect.
  warnings.warn("``reg_type`` no longer has any effect.", FutureWarning)
/net/argon/landclim2/lpierini/mesmer/mesmer/utils/select.py:67: FutureWarning: Passing `reg_dict` no longer has an effect.
  warnings.warn("Passing `reg_dict` no longer has an effect.", FutureWarning)
/net/argon/landclim2/lpierini/mesmer/mesmer/create_emulations/create_emus_gt.py:17: FutureWarning: 'create_emus_gt' has been renamed to `gather_gt_data`
  warnings.warn(


## Normalized Residuals and Emulations

In [85]:
dict_res_mm = {}
#dict_res_0_mm = {}

In [86]:
#CAREFUL: I need the predictors from the corresponding models!!!


esmodel = 'ACCESS-CM2'
#esmodel = 'CanESM5'
#esmodel = 'CESM2'

# --------------------------------------------------------------
# preparing data
# --------------------------------------------------------------
(
    time,
    PRED,
    SUB_PRED,
    reg_dict,
    ls,
    wgt_g,
    lon,
    lat,
    land_targ,
    land_pred,
    phi_gc,
    #ind,
    #gp2reg,
    #ww_reg,
    used_esms,
    dico_gps_nan,
) = load_inputs_MESMERx(cfg, [targ, pred, sub_pred], [esmodel])


# --------------------------------------------------------------
#DATA FORMATTING
# --------------------------------------------------------------

# esm test
esm = esmodel

# variable
land_targ_s, time_s = separate_hist_future(land_targ[esm], time[esm], cfg)

# predictor GMT at t
pred_s, time_s = separate_hist_future(PRED[esm], time[esm], cfg)

#Global trend only of GMT
preds_gt = {"time": time[esm]}
params_gt_pred = train_gt(PRED[esm], 'tas', esm, time[esm], cfg, save_params=False)

gt_pred_s = create_emus_gt(
    params_gt_pred, preds_gt, cfg, concat_h_f=False, save_emus=False
)


pred_s_m = {ss: np.tile(gt_pred_s[ss], (land_targ_s[ss].shape[0], 1)) for ss in gt_pred_s.keys()} #smoothed GMT, same for every member

# predictor GMT at t-1
tmp_pred_s = {
    scen: np.hstack([pred_s_m[scen][:, 0, np.newaxis], pred_s_m[scen][:, :-1]])
    for scen in pred_s_m.keys()
}
#tmp_pred_s, _ = separate_hist_future(tmp_pred, time[esm], cfg)

#global trend emulations
emus_gt_pred = create_emus_gt(
            params_gt_pred, preds_gt, cfg, concat_h_f=True, save_emus=False
        )

#global variability

gv_novolc_pred = {scen: PRED[esm][scen] - emus_gt_pred[scen] for scen in emus_gt_pred.keys()}
gv_novolc_pred_s, time_s = separate_hist_future(gv_novolc_pred, time[esm], cfg)


# preparing predictors (GMT)
predictors = []
for scen in pred_s.keys():
    predictors.append((xr.Dataset(), scen))
    predictors[-1][0]["GMT_t"] = xr.DataArray(
        pred_s_m[scen]-np.nanmin(pred_s_m['hist']),   #I want to start from 0 anomaly (that's how i took the training)
        coords={"member": np.arange(pred_s[scen].shape[0]), "time": time_s[scen]},
        dims=(
            "member",
            "time",
        ),
    )
    predictors[-1][0]["GMTv_t"] = xr.DataArray(
        gv_novolc_pred_s[scen],
        coords={"member": np.arange(pred_s[scen].shape[0]), "time": time_s[scen]},
        dims=(
            "member",
            "time",
        ),
    )
    #predictors[-1][0]["GMT_tm1"] = xr.DataArray(
    #    tmp_pred_s[scen],
    #    coords={"member": np.arange(pred_s[scen].shape[0]), "time": time_s[scen]},
   #     dims=(
    #        "member",
    #        "time",
    #    ),
    #)

# preparing target
target = []
for scen in land_targ_s.keys():
    target.append((xr.Dataset(), scen))
    target[-1][0][targ] = xr.DataArray(
        land_targ_s[scen],
        coords={
            "member": np.arange(land_targ_s[scen].shape[0]),
            "time": time_s[scen],
            "gridpoint": np.arange(land_targ_s[scen].shape[2]),
        },
        dims=(
            "member",
            "time",
            "gridpoint",
        ),
    )

#MAT01
xr_coeffs_distrib = xr_coeffs_distrib_rect_mix_glob.sel(model = esmodel)

xr_qual = xr_qual_rect_mix_glob.sel(model = esmodel)

expression_fit = mesmer_x_train.Expression(expr, expr_lab)
#distrib = expression_fit.evaluate(xr_coeffs_distrib, predictors[1][0], forced_shape=target[1][0]['rx1day'].dims)

xr_coeffs_distrib_m = xr_coeffs_distrib.drop_vars(expression_fit.coefficients_list)
xr_coeffs_distrib_mix = xr_coeffs_distrib_m.rename({var: var[:-1] for var in xr_coeffs_distrib_m.data_vars if var.endswith('m')})

#xr_coeffs_distrib_bs = xr_coeffs_distrib_no_rm.sel(model = esmodel)
#xr_qual_bs = xr_qual_no_rm.sel(model = esmodel)

transf_target = mesmer_x_train_utils.probability_integral_transform(  # noqa: F841
    data=target, 
    target_name = targ,
    expr_start=expr,
    expr_start_mix=expr_mix,
    coeffs_start=xr_coeffs_distrib,
    coeffs_start_mix=xr_coeffs_distrib_mix,
    qual_start = xr_qual,
    preds_start=predictors,
    expr_end="norm(loc=0, scale=1)",
)


#transf_target_0 = mesmer_x_train_utils.probability_integral_transform(  # noqa: F841
#    data=target, 
#    target_name = targ,
#    expr_start=expr,
#    coeffs_start=xr_coeffs_distrib_bs,
#    preds_start=predictors,
#    expr_end="norm(loc=0, scale=1)",
#)

#Transforming to dictionary containing scenarios

transf_target_dict = {targ: 
                      {str(transf_target[i][1]): transf_target[i][0] for i in range(len(transf_target))}
                     }
#transf_target_dict_0 = {targ: 
#                      {str(transf_target_0[i][1]): transf_target_0[i][0] for i in range(len(transf_target_0))}
#                     }

dict_resid = {}
#dict_resid_0 = {}

#maybe save only the residuals for historical and ssp585 as netcdf to save space
# Iterate through the outer dictionary (rx1day)
for scenario, np_array in transf_target_dict['rx1day'].items():
    # Assuming the shape is (member, time, gridpoint)
    members, timesteps, gridpoints = np_array.shape
    
    # Create DataArray with dimensions and coordinates
    dict_resid[scenario] = xr.DataArray(
        np_array,
        dims=['member', 'time', 'gridpoint'],
        coords={
            'member': np.arange(members),
            'time': np.arange(timesteps),
            'gridpoint': np.arange(gridpoints)
        },
        name=scenario
    )

#NO RECT NO MIX case
#for scenario, np_array in transf_target_dict_0['rx1day'].items():
    # Assuming the shape is (member, time, gridpoint)
#    members, time, gridpoints = np_array.shape
    
    # Create DataArray with dimensions and coordinates
#    dict_resid_0[scenario] = xr.DataArray(
#        np_array,
#        dims=['member', 'time', 'gridpoint'],
#        coords={
#            'member': np.arange(members),
#            'time': np.arange(time),
#            'gridpoint': np.arange(gridpoints)
#        },
#        name=scenario
#    )

#dict_res_mm[esmodel] = dict_resid
#dict_res_0_mm[esmodel] = dict_resid_0

dict_res_mm[esmodel] = dict_resid

/net/argon/landclim2/lpierini/mesmer/mesmer/io/load_constant_files.py:178: FutureWarning: ``reg_type`` no longer has any effect.
  warnings.warn("``reg_type`` no longer has any effect.", FutureWarning)
/net/argon/landclim2/lpierini/mesmer/mesmer/utils/select.py:67: FutureWarning: Passing `reg_dict` no longer has an effect.
  warnings.warn("Passing `reg_dict` no longer has an effect.", FutureWarning)
/net/argon/landclim2/lpierini/mesmer/mesmer/create_emulations/create_emus_gt.py:17: FutureWarning: 'create_emus_gt' has been renamed to `gather_gt_data`
  warnings.warn(


In [ ]:
for key, array in transf_target_dict['rx1day'].items():
    if np.isinf(array).any():
        print(f"inf found in {key}", flush=True)
    else:
        print('DICT OK!')
    if np.isnan(array).any():
        print(f"Nan found in {key}", flush=True)
    else:
        print('DICT OK!')

#### Emulations

In [ ]:
#from joblib import Parallel, delayed
import copy
import sys

sys.stdout = open('output_emulations_3.txt', 'w')

chosen_scenarios = ['h-ssp585', 'h-ssp370', 'h-ssp245', 'h-ssp126', 'h-ssp119']
#chosen_scenarios = ['h-ssp585']

ds_emus_dict = {}
#ds_emus_norm_dict = {}


transf_target_dict;
#transf_target_dict_no_rm = transf_target_dict_0;

print(esmodel)

#Number of emulations
n_members = 500

cfg.nr_emus_v = n_members
#num_cores = 10
phi_gc_dict = phi_gc 

xr_coeffs_distrib = xr_coeffs_distrib_rect_mix_glob.sel(model = esmodel)
xr_qual = xr_qual_rect_mix_glob.sel(model = esmodel)

expression_fit = mesmer_x_train.Expression(expr, expr_lab)

xr_coeffs_distrib_m = xr_coeffs_distrib.drop_vars(expression_fit.coefficients_list)
xr_coeffs_distrib_mix = xr_coeffs_distrib_m.rename({var: var[:-1] for var in xr_coeffs_distrib_m.data_vars if var.endswith('m')})

# training of parameters for Ar1 with SCI (step 1c)3
params_lv = train_lv(preds={}, targs = transf_target_dict, esm=esm, cfg=cfg, save_params=False, aux={'phi_gc': phi_gc_dict}, params_lv={})

#global trend emulations
emus_gt_pred = create_emus_gt(
            params_gt_pred, preds_gt, cfg, concat_h_f=True, save_emus=False
        )

## train global variability (AR process)
params_gv_pred = train_gv(gv_novolc_pred_s, pred, esm, cfg, save_params=False)

## emulator of global variability of predictor for ALL historical & scenario ATtached
preds_gv = {"time": {"all": list(time[esm].items())[0][1] } }
emus_gv_pred = create_emus_gv(params_gv_pred, preds_gv, cfg, save_emus=False) #this as predictor

## merging global emulators of predictor
emus_g_pred = create_emus_g( emus_gt_pred, emus_gv_pred, params_gt_pred, params_gv_pred, cfg, save_emus=False ) 
#this only used to have the total GMT for that year, not used as input unless I train on (GT+GV)

target_scenarios = list(emus_gt_pred.keys())

for scen_x in chosen_scenarios:
    if scen_x in target_scenarios:
        idx_scen_x  = target_scenarios.index(scen_x)
        print(f"The scenario '{scen_x}' is at index {idx_scen_x} in target_scenarios.", flush=True)
    else:
        print(f"The scenario '{scen_x}' is not in target_scenarios, skipping.", flush=True)
        continue
    
    scen_numb = idx_scen_x
    print(target_scenarios[scen_numb][1], flush=True)
    
    GMT = emus_gt_pred[scen_x] - np.min(emus_gt_pred[scen_x])
    time_gmt = time[esm][scen_x]
    GMT = np.tile(GMT, (n_members, 1))  # Shape (1000, 100)

    # Prepare predictors
    preds_newscen = [(xr.Dataset(), 'scen_og')]
    preds_newscen[-1][0]["GMT_t"] = xr.DataArray(
        GMT,
        coords={"member": np.arange(n_members), "time": time_gmt},
        dims=("member", "time"),
    )

    preds_newscen[-1][0]["GMTv_t"] = xr.DataArray(
        emus_gv_pred['all'],
        coords={"member": np.arange(n_members), "time": time_gmt},
        dims=("member", "time"),
    )

    preds_newscen[-1][0]["GMTtot_t"] = xr.DataArray(
        emus_g_pred[scen_x]  - np.min(emus_gt_pred[scen_x]),
        coords={"member": np.arange(n_members), "time": time_gmt},
        dims=("member", "time"),
    )

    predictors_dict = {'GMT_t': {'all': GMT}, 'GMTv_t': {'all': emus_gv_pred}}

    start_time = tm.time()
    # Emulation
    transf_emus = create_emus_lv(params_lv=params_lv, preds_lv=predictors_dict, cfg=cfg, save_emus=False, submethod="")

    print('transf_emus done', flush=True)
    # End time
    end_time = tm.time()
    
    # Execution time in seconds
    elapsed_time = end_time - start_time
    print(f"Elapsed time: {elapsed_time:.6f} seconds")
    # Define coordinates
    timesteps = time_gmt
    members = np.arange(n_members)
    grid_points = np.arange(transf_emus['all'][targ].shape[2])

    # Convert to xarray Dataset
    transf_emus_ds = xr.Dataset(
        {
            targ: xr.DataArray(
                transf_emus['all'][targ],
                coords={"member": members,"time": timesteps, "gridpoint": grid_points},
                dims=("member","time", "gridpoint"),
            )
        }
    )
    
    print('Doing Emus')
    start_time = tm.time()
    # Emulation steps
    emus = mesmer_x_train_utils.probability_integral_transform(
        data=[[transf_emus_ds, 'all']],
        target_name=targ,
        expr_start="norm(loc=0, scale=1)",
        expr_end=expr,
        expr_end_mix=expr_mix,
        coeffs_end=xr_coeffs_distrib,
        coeffs_end_mix=xr_coeffs_distrib_mix,
        qual_end=xr_qual,
        preds_end=preds_newscen
    )

    print('Done rect mix')
    # End time
    end_time = tm.time()
    
    # Execution time in seconds
    elapsed_time = end_time - start_time
    print(f"Elapsed time: {elapsed_time:.6f} seconds", flush=True)

    # Convert to xarray Dataset
    emus_ds_rect_mix = xr.Dataset(
        {
            targ: xr.DataArray(
                emus[0][0],
                coords={"member": members, "time": timesteps, "gridpoint": grid_points},
                dims=("member", "time", "gridpoint"),
            )
        }
    )
        
    # for every scenario
    ds_emus_dict[scen_x] = emus_ds_rect_mix
    print(len(ds_emus_dict[scen_x].time), flush = True)
    #ds_emus_dict[scen_x].to_netcdf('/net/argon/landclim2/lpierini/emulations/rx1day/emus_glob_' + esmodel + '_' + scen_x + '.nc')
    

print('DONE', flush = True)
#sys.stdout.close()

/net/argon/landclim2/lpierini/mesmer/mesmer/core/utils.py:73: OptimizeWarning: First element is local minimum.
  warnings.warn("First element is local minimum.", OptimizeWarning)
/net/argon/landclim2/lpierini/mesmer/mesmer/create_emulations/create_emus_gt.py:17: FutureWarning: 'create_emus_gt' has been renamed to `gather_gt_data`
  warnings.warn(
/home/lpierini/.conda/envs/mesmer_dev/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1545: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(
